# Phase 4A — Minimal ASC Train-Val Smoke Run

This notebook runs a minimal train-validation smoke test for the ASC branch.

Goals of this phase:

1. load the Phase 2 ASC jsonl files;
2. reuse the shared tokenizer and label mapping from Phase 3;
3. construct a lightweight Dataset and DataLoader;
4. initialize a sequence-classification model;
5. verify one forward pass, one backward pass, and several optimizer steps;
6. run a tiny validation loop;
7. confirm that the current ASC pipeline is trainable.

This notebook is NOT meant for full experiments or performance tuning.
It is only a proof that the ASC branch is ready for real training.

GAO YU: Currently is 1e-6 & SGD. At this moment, I just want to test if it is workable, but for real training should not be 1e-6, also, it should be adam/W

In [1]:
import copy
import json
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

d:\anaconda3\envs\tpwng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1. Define paths and lightweight smoke-run settings

This smoke run intentionally uses:
- a small subset of training and validation data,
- a small batch size,
- a short number of optimization steps,
- no advanced tricks.

The goal is stability and interface verification, not benchmark quality.

In [2]:
DATA_DIR = Path("outputs_phase2")
META_DIR = Path("outputs_phase3")
OUT_DIR = Path("outputs_phase4_asc_smoke")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ASC_TRAIN_PATH = DATA_DIR / "asc_train.jsonl"
ASC_VAL_PATH   = DATA_DIR / "asc_val.jsonl"
ASC_TEST_PATH  = DATA_DIR / "asc_test.jsonl"

SPLIT_STATS_PATH = DATA_DIR / "split_stats_report.json"
ASC_SMOKE_PHASE3_PATH = META_DIR / "asc_smoke_report.json"

SMOKE_REPORT_PATH = OUT_DIR / "asc_min_train_val_smoke_report.json"

# Use the same backbone validated in Phase 3
BACKBONE_NAME = "microsoft/deberta-v3-base"

# Optional lighter fallback if needed later:
# BACKBONE_NAME = "microsoft/deberta-v3-small"

MAX_LEN = 128
BATCH_SIZE = 8
LR = 1e-6
WEIGHT_DECAY = 0.0
FREEZE_BACKBONE_FOR_SMOKE = True
TRAIN_CLASSIFIER_ONLY_FOR_SMOKE = True

# Keep this intentionally small
SMOKE_TRAIN_SIZE = 256
SMOKE_VAL_SIZE = 64
MAX_TRAIN_STEPS = 20
MAX_VAL_BATCHES = 5

ASC_LABEL2ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}
ASC_ID2LABEL = {v: k for k, v in ASC_LABEL2ID.items()}

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## Step 2. Load metadata from earlier phases

This step confirms that the current notebook is operating on the expected Phase 2 / Phase 3 outputs.

In [3]:
with open(SPLIT_STATS_PATH, "r", encoding="utf-8") as f:
    split_stats = json.load(f)

with open(ASC_SMOKE_PHASE3_PATH, "r", encoding="utf-8") as f:
    asc_phase3_report = json.load(f)

print("Phase 2 split stats:")
pprint(split_stats["asc_validation"])

print("\nPhase 3 ASC smoke report:")
pprint(asc_phase3_report)

Phase 2 split stats:
{'test_all_labels_valid': True,
 'test_records': 1445,
 'test_records_with_bad_span': 0,
 'test_records_with_bad_template': 0,
 'test_records_with_invalid_match_type': 0,
 'test_records_with_invalid_sentiment': 0,
 'train_all_labels_valid': True,
 'train_records': 11369,
 'train_records_with_bad_span': 0,
 'train_records_with_bad_template': 0,
 'train_records_with_invalid_match_type': 0,
 'train_records_with_invalid_sentiment': 0,
 'val_all_labels_valid': True,
 'val_records': 1417,
 'val_records_with_bad_span': 0,
 'val_records_with_bad_template': 0,
 'val_records_with_invalid_match_type': 0,
 'val_records_with_invalid_sentiment': 0}

Phase 3 ASC smoke report:
{'all_train_labels_valid': True,
 'backbone': 'microsoft/deberta-v3-base',
 'batch_shape_attention_mask': [4, 19],
 'batch_shape_input_ids': [4, 19],
 'batch_shape_labels': [4],
 'sample_encoded_len': 16,
 'sample_label_id': 1,
 'test_examples': 1445,
 'train_examples': 11369,
 'val_examples': 1417}


## Step 3. Load ASC jsonl files

These are the task-specific exports produced in Phase 2.
Each record should represent one (sentence, aspect, sentiment) sample.

In [4]:
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records

In [5]:
asc_train = load_jsonl(ASC_TRAIN_PATH)
asc_val   = load_jsonl(ASC_VAL_PATH)
asc_test  = load_jsonl(ASC_TEST_PATH)

print("ASC train/val/test sizes:", len(asc_train), len(asc_val), len(asc_test))

ASC train/val/test sizes: 11369 1417 1445


## Step 4. Validate that the ASC records contain the expected fields

In [6]:
required_fields = [
    "doc_id", "raw_id", "split", "sentence", "aspect",
    "sentiment", "char_from", "char_to", "match_type"
]

sample = asc_train[0]
missing = [f for f in required_fields if f not in sample]

print("Missing fields:", missing)
print("Field validation passed:", len(missing) == 0)
pprint(sample)

Missing fields: []
Field validation passed: True
{'aspect': 'SpiceJet',
 'char_from': 0,
 'char_to': 8,
 'doc_id': 'sentfin_000001',
 'input_text_template': 'SpiceJet to issue 6.4 crore warrants to promoters '
                        '[SEP] SpiceJet',
 'match_type': 'whole_word_unique',
 'raw_id': 1,
 'sentence': 'SpiceJet to issue 6.4 crore warrants to promoters',
 'sentiment': 'neutral',
 'split': 'train'}


## Step 5. Build tiny smoke subsets

We intentionally use only a small subset to minimize resource usage.
This is sufficient to verify trainability.

In [7]:
train_smoke = asc_train[:SMOKE_TRAIN_SIZE]
val_smoke = asc_val[:SMOKE_VAL_SIZE]

print("Smoke train size:", len(train_smoke))
print("Smoke val size:", len(val_smoke))

Smoke train size: 256
Smoke val size: 64


## Step 6. Load the tokenizer

We reuse the same tokenizer family already validated in Phase 3.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(BACKBONE_NAME)

print("Loaded tokenizer:", BACKBONE_NAME)
print("Pad token:", tokenizer.pad_token)
print("CLS token:", tokenizer.cls_token)
print("SEP token:", tokenizer.sep_token)

Loaded tokenizer: microsoft/deberta-v3-base
Pad token: [PAD]
CLS token: [CLS]
SEP token: [SEP]


## Step 7. Inspect one ASC sample after tokenization

This confirms that the sentence-aspect pair is encoded correctly and that the label mapping is valid.

In [9]:
asc_sample = train_smoke[0]

print("Sentence:", asc_sample["sentence"])
print("Aspect:", asc_sample["aspect"])
print("Sentiment:", asc_sample["sentiment"])
print("Label ID:", ASC_LABEL2ID[asc_sample["sentiment"]])

Sentence: SpiceJet to issue 6.4 crore warrants to promoters
Aspect: SpiceJet
Sentiment: neutral
Label ID: 1


In [10]:
encoded_sample = tokenizer(
    asc_sample["sentence"],
    asc_sample["aspect"],
    truncation=True,
    max_length=MAX_LEN,
    padding=False
)

encoded_tokens = tokenizer.convert_ids_to_tokens(encoded_sample["input_ids"])

print("Encoded length:", len(encoded_sample["input_ids"]))
print("First tokens:")
print(encoded_tokens[:40])

Encoded length: 16
First tokens:
['[CLS]', '▁Spice', 'Jet', '▁to', '▁issue', '▁6', '.', '4', '▁crore', '▁warrants', '▁to', '▁promoters', '[SEP]', '▁Spice', 'Jet', '[SEP]']


## Step 8. Define a minimal Dataset wrapper

This Dataset keeps the raw ASC records unchanged and leaves tokenization to the collate step.
That keeps the implementation simple and easy to inspect.

In [11]:
class ASCSmokeDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        return self.records[idx]

In [12]:
train_dataset = ASCSmokeDataset(train_smoke)
val_dataset = ASCSmokeDataset(val_smoke)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))

Train dataset size: 256
Val dataset size: 64


## Step 9. Define a simple collate function

Each batch is constructed as:
- tokenizer(sentence, aspect)
- padding to the longest example in the batch
- labels mapped to integer IDs

In [13]:
def asc_collate_fn(batch):
    encodings = []
    labels = []

    for ex in batch:
        enc = tokenizer(
            ex["sentence"],
            ex["aspect"],
            truncation=True,
            max_length=MAX_LEN,
            padding=False
        )
        encodings.append(enc)
        labels.append(ASC_LABEL2ID[ex["sentiment"]])

    batch_enc = tokenizer.pad(
        encodings,
        padding=True,
        return_tensors="pt"
    )
    batch_enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return batch_enc

## Step 10. Build DataLoaders

For the smoke run:
- training loader uses shuffle=True
- validation loader uses shuffle=False
- num_workers=0 for stability

In [14]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=asc_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=asc_collate_fn
)

## Step 11. Inspect one training batch

This verifies that:
- tensors are created correctly,
- batch dimensions are reasonable,
- labels are attached correctly.

In [15]:
train_batch = next(iter(train_loader))

print("input_ids shape:", train_batch["input_ids"].shape)
print("attention_mask shape:", train_batch["attention_mask"].shape)
print("labels shape:", train_batch["labels"].shape)
print("labels:", train_batch["labels"])

input_ids shape: torch.Size([8, 19])
attention_mask shape: torch.Size([8, 19])
labels shape: torch.Size([8])
labels: tensor([1, 0, 1, 1, 1, 1, 1, 1])


## Step 12. Load a minimal sequence-classification model

We use the same backbone family and set `num_labels=3`.
This is the first point where we verify that the ASC data really fits a trainable model.

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    BACKBONE_NAME,
    num_labels=3,
    id2label=ASC_ID2LABEL,
    label2id=ASC_LABEL2ID
)

if FREEZE_BACKBONE_FOR_SMOKE:
    for param in model.parameters():
        param.requires_grad = False

    for name, param in model.named_parameters():
        if TRAIN_CLASSIFIER_ONLY_FOR_SMOKE:
            if name.startswith("classifier"):
                param.requires_grad = True
        else:
            if name.startswith("classifier") or name.startswith("pooler"):
                param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Device:", device)
print("Model loaded:", BACKBONE_NAME)
print("Freeze backbone for smoke:", FREEZE_BACKBONE_FOR_SMOKE)
print("Train classifier only for smoke:", TRAIN_CLASSIFIER_ONLY_FOR_SMOKE)
print("Trainable params:", trainable_params)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 39572.68it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dens

Device: cuda
Model loaded: microsoft/deberta-v3-base
Freeze backbone for smoke: True
Train classifier only for smoke: True
Trainable params: 2307


## Step 13. Define a minimal optimizer

No scheduler and no complex training tricks are used here.
This is only a smoke run.

In [17]:
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR
)

## Step 14. Run one forward pass before training

This confirms that the batch can be consumed by the model and that the loss is computable.

In [18]:
batch_for_forward = {k: v.to(device) for k, v in train_batch.items()}

model.train()
outputs = model(**batch_for_forward)

print("Loss:", float(outputs.loss))
print("Logits shape:", outputs.logits.shape)

Loss: 1.0724539756774902
Logits shape: torch.Size([8, 3])


## Step 15. Run one backward pass

This confirms that gradients can flow and that the model is trainable.

In [19]:
optimizer.zero_grad()
outputs.loss.backward()
optimizer.zero_grad()

print("One backward pass completed successfully.")

One backward pass completed successfully.


## Step 16. Run a tiny training loop

This loop is intentionally short.
Its purpose is only to prove that:
- batches can be processed repeatedly,
- optimization steps can run,
- loss values remain finite.

In [20]:
# Start the tiny loop from a clean model state after the one-batch
# forward/backward smoke check above.
model = AutoModelForSequenceClassification.from_pretrained(
    BACKBONE_NAME,
    num_labels=3,
    id2label=ASC_ID2LABEL,
    label2id=ASC_LABEL2ID
)

if FREEZE_BACKBONE_FOR_SMOKE:
    for param in model.parameters():
        param.requires_grad = False

    for name, param in model.named_parameters():
        if TRAIN_CLASSIFIER_ONLY_FOR_SMOKE:
            if name.startswith("classifier"):
                param.requires_grad = True
        else:
            if name.startswith("classifier") or name.startswith("pooler"):
                param.requires_grad = True

model.to(device)

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR
)

last_finite_state = copy.deepcopy(model.state_dict())

train_losses = []
train_steps = 0

model.train()

for batch in train_loader:
    batch = {k: v.to(device) for k, v in batch.items()}

    optimizer.zero_grad()
    step_start_state = copy.deepcopy(model.state_dict())
    outputs = model(**batch)
    loss = outputs.loss

    if not torch.isfinite(loss) or not torch.isfinite(outputs.logits).all():
        print(f"Train step {train_steps + 1:02d} | non-finite loss encountered, stopping early.")
        break

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    params_finite = all(torch.isfinite(p).all() for p in model.parameters() if p.requires_grad)
    if not params_finite:
        model.load_state_dict(step_start_state)
        print(f"Train step {train_steps + 1:02d} | parameters became non-finite after update, reverting and stopping early.")
        break

    train_losses.append(float(loss.detach().cpu()))
    last_finite_state = copy.deepcopy(model.state_dict())
    train_steps += 1

    print(f"Train step {train_steps:02d} | loss = {train_losses[-1]:.4f}")

    if train_steps >= MAX_TRAIN_STEPS:
        break

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 28285.84it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dens

Train step 01 | loss = 1.1883
Train step 02 | loss = 1.1815
Train step 03 | loss = 1.1922
Train step 04 | loss = 1.1670
Train step 05 | loss = 1.1740
Train step 06 | loss = 1.1863
Train step 07 | loss = 1.1912
Train step 08 | loss = 1.1786
Train step 09 | loss = 1.1676
Train step 10 | loss = 1.1731
Train step 11 | loss = 1.1530
Train step 12 | loss = 1.1956
Train step 13 | loss = 1.1687
Train step 14 | loss = 1.1434
Train step 15 | loss = 1.1566
Train step 16 | loss = 1.1414
Train step 17 | loss = 1.1937
Train step 18 | loss = 1.1613
Train step 19 | loss = 1.1752
Train step 20 | loss = 1.1442


## Step 17. Run a tiny validation loop

This validation is also intentionally short.
The goal is only to verify that evaluation-mode inference works and that we can compute a simple metric.

In [21]:
# Restore the last known finite weights before validation so that
# evaluation remains runnable even if a later train step diverged.
model.load_state_dict(last_finite_state)

val_losses = []
val_correct = 0
val_total = 0
val_batches_seen = 0

model.eval()

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits

        if not torch.isfinite(loss) or not torch.isfinite(logits).all():
            print(f"Val batch {val_batches_seen + 1:02d} | non-finite loss encountered, stopping early.")
            break

        preds = torch.argmax(logits, dim=-1)
        labels = batch["labels"]

        val_losses.append(float(loss.detach().cpu()))
        val_correct += int((preds == labels).sum().item())
        val_total += int(labels.numel())
        val_batches_seen += 1

        print(f"Val batch {val_batches_seen:02d} | loss = {val_losses[-1]:.4f}")

        if val_batches_seen >= MAX_VAL_BATCHES:
            break

Val batch 01 | loss = 1.1786
Val batch 02 | loss = 1.1771
Val batch 03 | loss = 1.1778
Val batch 04 | loss = 1.1778
Val batch 05 | loss = 1.1790


## Step 18. Summarize the smoke-run outcome

At this stage, low accuracy is NOT a problem.
The important thing is that training and validation both complete successfully.

In [22]:
train_loss_mean = float(np.mean(train_losses)) if len(train_losses) > 0 else None
val_loss_mean = float(np.mean(val_losses)) if len(val_losses) > 0 else None
val_acc = float(val_correct / val_total) if val_total > 0 else None

print("Mean train loss:", train_loss_mean)
print("Mean val loss:", val_loss_mean)
print("Val accuracy:", val_acc)

Mean train loss: 1.1716544926166534
Mean val loss: 1.1780611515045165
Val accuracy: 0.0


## Step 19. Save the ASC minimal smoke-run report

In [23]:
smoke_report = {
    "backbone": BACKBONE_NAME,
    "device": str(device),
    "train_records_total": len(asc_train),
    "val_records_total": len(asc_val),
    "test_records_total": len(asc_test),
    "smoke_train_size": len(train_smoke),
    "smoke_val_size": len(val_smoke),
    "batch_size": BATCH_SIZE,
    "max_len": MAX_LEN,
    "freeze_backbone_for_smoke": FREEZE_BACKBONE_FOR_SMOKE,
    "train_classifier_only_for_smoke": TRAIN_CLASSIFIER_ONLY_FOR_SMOKE,
    "max_train_steps": MAX_TRAIN_STEPS,
    "max_val_batches": MAX_VAL_BATCHES,
    "one_batch_input_shape": list(train_batch["input_ids"].shape),
    "one_batch_label_shape": list(train_batch["labels"].shape),
    "forward_pass_ok": True,
    "backward_pass_ok": True,
    "train_steps_completed": train_steps,
    "val_batches_completed": val_batches_seen,
    "mean_train_loss": train_loss_mean,
    "mean_val_loss": val_loss_mean,
    "val_accuracy": val_acc
}

with open(SMOKE_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(smoke_report, f, ensure_ascii=False, indent=2)

pprint(smoke_report)
print("Saved smoke report to:", SMOKE_REPORT_PATH)

{'backbone': 'microsoft/deberta-v3-base',
 'backward_pass_ok': True,
 'batch_size': 8,
 'device': 'cuda',
 'forward_pass_ok': True,
 'freeze_backbone_for_smoke': True,
 'max_len': 128,
 'max_train_steps': 20,
 'max_val_batches': 5,
 'mean_train_loss': 1.1716544926166534,
 'mean_val_loss': 1.1780611515045165,
 'one_batch_input_shape': [8, 19],
 'one_batch_label_shape': [8],
 'smoke_train_size': 256,
 'smoke_val_size': 64,
 'test_records_total': 1445,
 'train_classifier_only_for_smoke': True,
 'train_records_total': 11369,
 'train_steps_completed': 20,
 'val_accuracy': 0.0,
 'val_batches_completed': 5,
 'val_records_total': 1417}
Saved smoke report to: outputs_phase4_asc_smoke\asc_min_train_val_smoke_report.json


## Step 20. Evaluate whether the ASC smoke run is successful

This smoke run is considered successful if:
- data loads correctly,
- tokenizer works,
- batch tensors are created,
- model forward works,
- backward works,
- a few train steps complete,
- a few validation batches complete.

In [24]:
import pandas as pd
checklist = [
    ("ASC train file loaded", len(asc_train) > 0),
    ("ASC val file loaded", len(asc_val) > 0),
    ("Tokenizer loaded", tokenizer is not None),
    ("One train batch created", train_batch["input_ids"].shape[0] > 0),
    ("Forward pass completed", True),
    ("Backward pass completed", True),
    ("At least one train step completed", train_steps > 0),
    ("At least one val batch completed", val_batches_seen > 0),
    ("Smoke report saved", SMOKE_REPORT_PATH.exists()),
]

check_df = pd.DataFrame(checklist, columns=["check_item", "passed"])
display(check_df)

print("ASC MINIMAL SMOKE RUN COMPLETE:", check_df["passed"].all())

,check_item,passed
0,ASC train file loaded,True
1,ASC val file loaded,True
2,Tokenizer loaded,True
3,One train batch created,True
4,Forward pass completed,True
5,Backward pass completed,True
6,At least one train step completed,True
7,At least one val batch completed,True
8,Smoke report saved,True


ASC MINIMAL SMOKE RUN COMPLETE: True


## Phase 4A conclusion

If all checks pass, then the current ASC pipeline has been verified to be trainable.

This means:
- the Phase 2 ASC exports are structurally correct,
- the Phase 3 tokenizer/interface logic is valid,
- model-facing preprocessing is sufficient,
- and the project is ready to move from interface validation to actual ASC experimentation.

The next logical step is:
1. either extend this into a real ASC baseline training notebook,
2. or repeat the same minimal smoke-run logic for the ATE branch.